### YOLOv8n detector and Resnet50 

Detection Accuracy: 72.30%

map:  0.2884

rank1: 0.2455

## To-Do List

1. **Assign IDs to `gallery_df`**
   - For each gallery detection, match it with the ground-truth bounding boxes using **IoU**.
   - Assign the corresponding **ground-truth person ID** based on the best IoU match.

2. **Evaluate Query–Gallery Matches**
   - For each query image, compare its **ground-truth ID** with the **IDs of the retrieved gallery images**.
   - Mark retrieved results as **positive** if the IDs match, otherwise **negative**.

3. **Compute Evaluation Metrics**
   - **mAP (mean Average Precision)**:
     - For each query, compute **AP** as the average of precision values at ranks where a positive match is retrieved.
     - Report **mAP** as the mean AP over all queries.
   - **Rank-1 Accuracy**:
     - Compute the fraction of queries for which the **top-1 retrieved result** is a positive match.

4. **Train and Evaluate with Metric Learning**
   - Use **`frame_train`** to train a metric learning model.
   - Use the **test set** to perform retrieval for query images and evaluate performance using mAP and Rank-1 accuracy.

### Imports and Configs

In [4]:
!pip install faiss-cpu


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 947.7 kB/s eta 0:00:0000:0100:01


In [114]:
# lets get started with single image
image_path = "../archive/frames"
import os
# sample random image from the folder
import random
from IPython.display import display
from PIL import Image
import time
from ultralytics import YOLO
import pandas as pd
import torch
from torchvision import transforms
import matplotlib.pyplot as plt
import cv2
from PIL import ImageDraw
%matplotlib inline
import numpy as np
from tqdm import tqdm
import torch.nn.functional as F
import faiss
from scipy.io import loadmat
from pandas import DataFrame


In [2]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
EMB_DIM = 2048
IMG_SIZE = (256, 128)
TOP_K = 10


In [3]:

# To reload the index and metadata later:
index = faiss.read_index("gallery_with_resnet50.index")
gallery_df = pd.read_csv("gallery_metadata_with_resnet50.csv")

### Load models

In [10]:
detector = YOLO("yolov8n.pt").to(DEVICE).eval()
reid_model = torch.hub.load('pytorch/vision:v0.10.0', "resnet50", weights="ResNet50_Weights.DEFAULT" ).to(DEVICE).eval()
reid_model.fc = torch.nn.Identity()  # Remove the final classification layer
reid_model = reid_model.to(DEVICE).eval()




Using cache found in /home/saad/.cache/torch/hub/pytorch_vision_v0.10.0


In [11]:
preprocess = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

### Index image paths for gallery and query images

In [112]:
def index_images(root):
    return [
        os.path.join(root, f)
        for root, _, files in os.walk(root)
        for f in files
        if f.endswith((".jpg", ".png"))
    ]

gallery_paths = index_images("../archive/frames")
gallery_paths = gallery_paths # Debug purpose
query_paths = index_images("../archive/query_box")
query_paths = query_paths



In [12]:

IMG_SIZE = (224, 224)  # Example size, adjust as needed

def detect_and_crop(img, detector, target_class=0):
    """
    Detect objects in the image and crop all instances of the target class (default: person).

    Args:
        img (np.ndarray): Input image.
        detector (YOLO): YOLO object detector.
        target_class (int or list): Class ID(s) to keep. Default 0 (person)

    Returns:
        crops (list): List of cropped images.
        det_info (list): List of detection info dictionaries.
    """
    detections = detector(img, verbose=False)
    crops = []
    det_info = []

    for det in detections:
        num_boxes = len(det.boxes)
        for i in range(num_boxes):
            bbox = det.boxes.xyxy[i].cpu().numpy()
            conf = det.boxes.conf[i].cpu().numpy()
            cls = int(det.boxes.cls[i].cpu().numpy())

            # Only keep target class(es)
            if (isinstance(target_class, int) and cls != target_class) or \
               (isinstance(target_class, list) and cls not in target_class):
                continue

            # Save detection info
            det_info.append({
                "x_min": bbox[0],
                "y_min": bbox[1],
                "x_max": bbox[2],
                "y_max": bbox[3],
                "confidence": conf,
                "class": cls,
            })

            # Crop and resize
            crop = img[int(bbox[1]):int(bbox[3]), int(bbox[0]):int(bbox[2])]
            if crop.size == 0:
                continue
            crop = cv2.resize(crop, IMG_SIZE)
            crops.append(crop)

    return crops, det_info


In [15]:
def extract_embeddings(crops, model):

    """Extract embeddings from cropped images using the re-identification model.
    Args:
        crops (list): List of cropped images.
        model (torch.nn.Module): The re-identification model.
    Returns:
        emb (numpy.ndarray): Array of embeddings.
    """

    # Ensure batch dimension and consistent array type
    if isinstance(crops, np.ndarray) and crops.ndim == 3:
        crops = crops[None, ...]  # add batch dim for single image
    elif isinstance(crops, list):
        crops = np.stack(crops, axis=0)  # list -> (B, H, W, C)

    crops = torch.tensor(crops).permute(0,3,1,2).float().to(DEVICE) / 255.
    with torch.no_grad():
        emb = model(crops)
        if isinstance(emb, (list, tuple)):
            emb = emb[0]
        emb = F.normalize(emb, dim=1)
    return emb.cpu().numpy()


In [72]:
columns = [
    "image_path",
    "x_min", "y_min", "x_max", "y_max",
    "confidence",
    "class",
    "faiss_id"
]

gallery_rows = []

index = faiss.IndexFlatIP(EMB_DIM)


### Build gallery (single loop, clean)

In [74]:
for path in tqdm(gallery_paths):
    img = cv2.imread(path)
    if img is None:
        continue

    crops, det_info = detect_and_crop(img, detector)

    if len(crops) == 0:
        continue

    embeddings = extract_embeddings(crops, reid_model)
    # print(embeddings.shape)

    index.add(embeddings)
    faiss_ids = np.arange(index.ntotal - len(embeddings), index.ntotal)

    for info, fid in zip(det_info, faiss_ids):
        gallery_rows.append([
            path,
            info["x_min"],
            info["y_min"],
            info["x_max"],
            info["y_max"],
            info["confidence"],
            info["class"],
            fid
        ])


100%|██████████| 11816/11816 [1:40:56<00:00,  1.95it/s] 


In [4]:
gallery_df

,image_path,x_min,y_min,x_max,y_max,confidence,class,faiss_id
0,../archive/frames/c5s1_067023.jpg,1696.411000,298.03885,1835.03390,579.92840,0.836966,0,0
1,../archive/frames/c5s1_067023.jpg,1158.350200,332.11346,1214.69910,495.46353,0.738274,0,1
2,../archive/frames/c5s1_067023.jpg,355.437300,343.48224,438.21450,517.09040,0.724844,0,2
3,../archive/frames/c5s1_067023.jpg,475.557070,353.94820,512.74140,436.78366,0.702278,0,3
4,../archive/frames/c5s1_067023.jpg,771.640000,338.98407,820.67860,453.39377,0.534687,0,4
...,...,...,...,...,...,...,...,...
52920,../archive/frames/c1s1_048101.jpg,23.347492,495.79388,132.92989,681.92570,0.461277,0,52920
52921,../archive/frames/c1s4_005556.jpg,1474.673600,442.67030,1711.02870,1075.30190,0.886001,0,52921
52922,../archive/frames/c1s4_005556.jpg,108.053925,414.00010,348.16290,1075.43540,0.819178,0,52922
52923,../archive/frames/c1s4_005556.jpg,480.861600,476.80392,535.91730,579.57440,0.416791,0,52923


### Save the fais index

In [ ]:
faiss.write_index(index, "gallery_with_resnet50.index")
gallery_df = pd.DataFrame(gallery_rows, columns=columns)
gallery_df.to_csv("gallery_metadata_with_resnet50.csv", index=False)


# To reload the index and metadata later:
# index = faiss.read_index("gallery_with_resnet50.index")
# gallery_df = pd.read_csv("gallery_metadata_with_resnet50.csv")


In [118]:

def iou_vec(box, boxes):
    """
    box:   (4,)  -> (x, y, w, h)
    boxes: (N,4)-> (x, y, w, h)
    returns: (N,) IoU values
    """
    x, y, w, h = box
    x2, y2 = x + w, y + h

    boxes = boxes.astype(float)
    bx, by, bw, bh = boxes.T
    bx2, by2 = bx + bw, by + bh

    inter_x1 = np.maximum(x, bx)
    inter_y1 = np.maximum(y, by)
    inter_x2 = np.minimum(x2, bx2)
    inter_y2 = np.minimum(y2, by2)

    inter_w = np.clip(inter_x2 - inter_x1, 0, None)
    inter_h = np.clip(inter_y2 - inter_y1, 0, None)
    inter_area = inter_w * inter_h

    area_box = w * h
    area_boxes = bw * bh

    union = area_box + area_boxes - inter_area
    return np.where(union > 0, inter_area / union, 0)


# Assign person IDs to gallery detections based on .mat annotations and IoU matching

person_ids = []
mat_cache = {}

for _, row in gallery_df.iterrows():
    path = row['image_path']
    filename = os.path.splitext(os.path.basename(path))[0]
    mat_path = os.path.join('../archive/annotations/', filename + '.jpg.mat')

    # Load .mat once per image
    if mat_path not in mat_cache:
        try:
            data = loadmat(mat_path)
            box_new = data['box_new']

            mat_cache[mat_path] = {
                "boxes": np.array([[b[1], b[2], b[3], b[4]] for b in box_new]),
                "ids":   np.array([int(b[0]) for b in box_new])
            }
        except Exception:
            mat_cache[mat_path] = {
                "boxes": np.empty((0, 4)),
                "ids":   np.empty((0,))
            }

    x_min, y_min, x_max, y_max = (
        row['x_min'], row['y_min'], row['x_max'], row['y_max']
    )
    w, h = x_max - x_min, y_max - y_min
    query_box = np.array([x_min, y_min, w, h])

    ann = mat_cache[mat_path]

    if ann["boxes"].shape[0] == 0:
        person_ids.append(-1)
    else:
        ious = iou_vec(query_box, ann["boxes"])
        best = np.argmax(ious)
        person_ids.append(
            ann["ids"][best] if ious[best] > 0.5 else -1
        )

gallery_df['person_id'] = person_ids


In [119]:
gallery_df

,image_path,x_min,y_min,x_max,y_max,confidence,class,faiss_id,person_id
0,../archive/frames/c5s1_067023.jpg,1696.411000,298.03885,1835.03390,579.92840,0.836966,0,0,258
1,../archive/frames/c5s1_067023.jpg,1158.350200,332.11346,1214.69910,495.46353,0.738274,0,1,254
2,../archive/frames/c5s1_067023.jpg,355.437300,343.48224,438.21450,517.09040,0.724844,0,2,233
3,../archive/frames/c5s1_067023.jpg,475.557070,353.94820,512.74140,436.78366,0.702278,0,3,206
4,../archive/frames/c5s1_067023.jpg,771.640000,338.98407,820.67860,453.39377,0.534687,0,4,248
...,...,...,...,...,...,...,...,...,...
52920,../archive/frames/c1s1_048101.jpg,23.347492,495.79388,132.92989,681.92570,0.461277,0,52920,-2
52921,../archive/frames/c1s4_005556.jpg,1474.673600,442.67030,1711.02870,1075.30190,0.886001,0,52921,633
52922,../archive/frames/c1s4_005556.jpg,108.053925,414.00010,348.16290,1075.43540,0.819178,0,52922,668
52923,../archive/frames/c1s4_005556.jpg,480.861600,476.80392,535.91730,579.57440,0.416791,0,52923,-2


Detection Accuracy: 72.30%


### Query pipeline

In [104]:
def query_image(path, reid_model, index, gallery_df, k=5):
    img = cv2.imread(path)
    img = cv2.resize(img, IMG_SIZE)

    emb = extract_embeddings([img], reid_model)
    D, I = index.search(emb, k)   # -> D: distances, I: indices

    results = gallery_df.iloc[I[0]].copy()
    results["similarity"] = D[0]

    path = path.replace("query_box", "frames")
    filename = os.path.basename(path)
    query_frame = filename.split("_", 1)[1]
    query_frame_path = os.path.join(os.path.dirname(path), query_frame)
    results = results[results["image_path"] != query_frame_path]


    results = results[results["image_path"] != query_frame_path]
    return results


In [110]:
def compute_ap(results, query_path):
    """
    results: DataFrame sorted by similarity descending
    query_id: identity of the query image

    """
        
    # Get the filename
    filename = os.path.basename(query_path)  # "931_c5s2_041530.jpg"

    # Extract the part before the first underscore
    query_id = int(filename.split("_")[0])  # 931
    
    y_true = (results['person_id'] == query_id).astype(int).values
    y_score = results['similarity'].values

    # If no correct match, AP = 0
    if y_true.sum() == 0:
        return 0.0

    # Compute average precision
    from sklearn.metrics import average_precision_score
    return average_precision_score(y_true, y_score)

def rank1(results, query_path):
    filename = os.path.basename(query_path)
    query_id = int(filename.split("_")[0])
    top1_id = results.iloc[0]['person_id']
    return int(top1_id == query_id)



### Compute map and Rank-1 on all query images

In [113]:
ap = []
rank1_scores = []
for query_path in query_paths:
    results = query_image(query_path, reid_model, index, gallery_df) 
    # print("Query Image:", query_path)
    results_sorted = results.sort_values("similarity", ascending=False)
    # display(results_sorted.head(TOP_K))
    ap_value = compute_ap(results_sorted, query_path)
    ap.append(ap_value)
    r1 = rank1(results_sorted, query_path)
    rank1_scores.append(r1)

mean_ap = np.mean(ap)
mean_rank1 = np.mean(rank1_scores)
print(f"Mean Average Precision (mAP): {mean_ap:.4f}")
print(f"Mean Rank-1 Accuracy: {mean_rank1:.4f}")

Mean Average Precision (mAP): 0.2884
Mean Rank-1 Accuracy: 0.2455


In [133]:
gallery_df['person_id'].value_counts()
# How many are -1s?
no_detectios = (gallery_df['person_id'] == -1).sum()
total_detections = gallery_df.shape[0]
print("Detection Accuracy: {:.2f}%".format((1 - no_detectios/total_detections) * 100))

Detection Accuracy: 72.30%


Comments:
These are the baseline results we get by using renset50 as retrieval model. Since its trianed on imagenet and not on the person re-id purpose so results are not very good.
Also YOLOv8 gives a detection accuracy of 72.30%. This can be made better by using higher models of this series or may be a two stage detector